# Chapter 05 · Notebook 03: Two input features

[Chapter guide](README.md) · [Previous notebook](02_training_layer.ipynb) · [Course index](../README.md)

Previously, each example contained one input value. Now each example contains two features, and the model still produces one prediction. A feature is one input value describing an example.

Our rule is now:

`prediction = weight₁ × input₁ + weight₂ × input₂ + bias`

There are three learned parameter values: two weights and one bias. These same values are shared across examples. We first set values ourselves to inspect predictions, then calculate gradients and train the layer.

## Contents

1. [Inspect the parameters](#parameters)
2. [Predict for one example](#one-example)
3. [Predict for four examples](#four-examples)
4. [Calculate a gradient for each weight](#feature-gradients)
5. [Change which feature is active](#second-feature-gradient)
6. [Update the parameters with SGD](#two-weight-update)
7. [Combine examples into a batch](#batch-gradients)
8. [Apply the batch update](#batch-update)
9. [Train on all four examples](#full-training)
10. [Evaluate unseen validation examples](#validation)

Work one lesson at a time with the mentoring conversation. Run Setup in a fresh kernel; this notebook has no dependency on another notebook's variables.

## Setup: import PyTorch

We will use the same linear layer type as before, with two input features instead of one.


In [ ]:
import torch  # PyTorch supplies the layer and tensor operations.
print("PyTorch version:", torch.__version__)


<a id="parameters"></a>
## Lesson 01: inspect two weights and one bias

`torch.nn.Linear(2, 1)` accepts two input values **per example** and produces one output value **per example**. The `2` is the feature count, not the batch size.

The weight tensor has shape `[1, 2]`: one output, with a weight for each of two input features. Its two entries are `weight₁` and `weight₂`. The bias has shape `[1]`: one bias for that output. Together they contain three scalar parameter values.

Creating the layer initializes values but does not train them. In the next lesson, we will explicitly assign known values.

**Before running:** what weight shape, bias shape, and total parameter count do you expect?

**My prediction:** _write here_


In [ ]:
# Two input features per example, one prediction per example.
model = torch.nn.Linear(in_features=2, out_features=1)

# Count scalar values, rather than the number of parameter tensors.
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print("Layer:", model)
print("Weight shape:", model.weight.shape)
print("Bias shape:", model.bias.shape)
print("Total learned parameter values:", parameter_count)


**My observation and explanation:** _write here_

<a id="one-example"></a>
## Lesson 02: predict for one example with two features

Use our conversation's example: inputs `[2, 4]`, weights `[3, 1]`, and bias `2`. The prediction should be `(3 × 2) + (1 × 4) + 2 = 12`.

The input has shape `[1, 2]`: one example, two features. The output has shape `[1, 1]`: one example, one prediction. We set parameter values inside `no_grad()` because this is explicit initialization.

**Before running:** write the calculation for this prediction.

**My calculation:** _write here_


In [ ]:
# Set the two weights in feature order and the single output bias.
with torch.no_grad():
    model.weight.copy_(torch.tensor([[3., 1.]]))
    model.bias.fill_(2.)

# One row is one example; its two columns are input₁ and input₂.
single_x = torch.tensor([[2., 4.]])
with torch.no_grad():
    prediction = model(single_x)

print("Weights:", model.weight.detach().flatten().tolist())
print("Bias:", model.bias.item())
print("Input shape:", single_x.shape)
print("Output shape:", prediction.shape)
print("Layer prediction:", prediction.item())
print("Manual prediction:", 3 * 2 + 1 * 4 + 2)


**My observation and explanation:** _write here_

<a id="four-examples"></a>
## Lesson 03: predict for four examples with the same parameters

The input tensor now has shape `[4, 2]`: four examples, two features each. Each row produces one prediction, so the output shape is `[4, 1]`.

All four rows use the same two weights and one bias. The model does not create more parameters when the batch contains more examples. This cell only makes predictions; it performs no training update.

**Before running:** predict the output for each row using `3 × input₁ + input₂ + 2`.

**My predictions:** _write here_


In [ ]:
# Each row is an example; the columns keep the same feature order.
x = torch.tensor([[0., 0.], [1., 0.], [0., 1.], [2., 4.]])

# Use the existing weights and bias to predict for all four examples.
with torch.no_grad():
    predictions = model(x)
    manual_predictions = 3 * x[:, 0:1] + x[:, 1:2] + 2

print("Input shape:", x.shape)
print("Output shape:", predictions.shape)
print("Predictions:", predictions.flatten().tolist())
print("Matches manual calculation:", torch.allclose(predictions, manual_predictions))
print("Total learned parameter values:", sum(p.numel() for p in model.parameters()))


**My observation and explanation:** _write here_


<a id="feature-gradients"></a>
## Lesson 04: calculate a gradient for each weight

For input `[1, 0]`, the prediction is `weight₁ + bias`. The second weight is multiplied by zero, so changing it cannot change this example's prediction. Its loss gradient is therefore zero for this example. It can have a nonzero gradient on other examples where the second input is nonzero.

Our previous layer had the correct weights already. For this learning experiment, create a separate layer called `training_model`, with starting weights `[1, 1]` and bias `0`. This preserves the prediction examples above and gives us a model that makes an error.

Use input `[1, 0]` and supplied target `5`, matching our rule `3 × input₁ + input₂ + 2`. The starting prediction is `1` and the squared loss is `16`.

Calculate loss with `MSELoss`, then call `backward()`. The weight gradients have the same shape as the weights, `[1, 2]`; the printout flattens them for readability. Their order corresponds to the two input features. Backward still does not update parameters.

**Before running:** which weight gradient should be zero? Will the weights and bias change yet?

**My prediction:** _write here_


In [ ]:
# Start a separate learning experiment with known, imperfect parameters.
training_model = torch.nn.Linear(2, 1)
with torch.no_grad():
    training_model.weight.fill_(1.)  # Both weights start at 1.
    training_model.bias.fill_(0.)

# One example: feature 1 is active and feature 2 is zero.
example_x = torch.tensor([[1., 0.]])
example_y = torch.tensor([[5.]])
loss_fn = torch.nn.MSELoss()

training_model.zero_grad(set_to_none=True)  # Clear gradients without changing values.
prediction = training_model(example_x)
loss = loss_fn(prediction, example_y)
loss.backward()  # Calculate each parameter's gradient; no update yet.

print("Prediction:", prediction.item())
print("Loss:", loss.item())
print("Weight gradients [weight1, weight2]:", training_model.weight.grad.flatten().tolist())
print("Bias gradient:", training_model.bias.grad.item())
print("Weights still:", training_model.weight.detach().flatten().tolist())
print("Bias still:", training_model.bias.item())


**My observation and explanation:** _write here_

The zero gradient here applies to this example at the current calculation. It does not mean the second weight is never trainable. The next experiment will use a nonzero second feature.


<a id="second-feature-gradient"></a>
## Lesson 05: change which feature is active

Now use input `[0, 1]` and target `3`, again from our rule `3 × input₁ + input₂ + 2`. Keep the starting weights `[1, 1]` and bias `0` so the first prediction is still `1`.

The first input is zero, so the first weight cannot affect this example's prediction. The second input is one, so the second weight and bias both affect it. Reset parameters explicitly to make the comparison repeatable, clear gradients, and calculate a fresh loss and gradients for this new example.

**Before running:** which weight gradient will be zero? Why can the same weight have a nonzero gradient in one example and zero in another?

**My prediction and explanation:** _write here_


In [ ]:
# Use the same starting parameters as Lesson 04 for a fair comparison.
with torch.no_grad():
    training_model.weight.fill_(1.)
    training_model.bias.fill_(0.)

# This time feature 1 is zero and feature 2 is active.
example_x = torch.tensor([[0., 1.]])
example_y = torch.tensor([[3.]])

training_model.zero_grad(set_to_none=True)  # Remove the previous example's gradients.
prediction = training_model(example_x)     # Calculate a fresh prediction.
loss = loss_fn(prediction, example_y)
loss.backward()                            # Calculate fresh gradients; no update yet.

print("Prediction:", prediction.item())
print("Loss:", loss.item())
print("Weight gradients [weight1, weight2]:", training_model.weight.grad.flatten().tolist())
print("Bias gradient:", training_model.bias.grad.item())
print("Weights still:", training_model.weight.detach().flatten().tolist())
print("Bias still:", training_model.bias.item())


**My observation and explanation:** _write here_

Compare the two examples at the same starting parameters:

| Input | Target | Weight 1 gradient | Weight 2 gradient | Bias gradient |
|---|---:|---:|---:|---:|
| `[1, 0]` | 5 | -8 | 0 | -8 |
| `[0, 1]` | 3 | 0 | -4 | -4 |

The gradient depends on the example as well as the current parameter values. Neither weight is permanently inactive. On a batch of examples, the gradients combine the contributions of those examples according to the loss calculation.


<a id="two-weight-update"></a>
## Lesson 06: update the parameters using the new gradients

Lesson 05 calculated gradients for input `[0, 1]` and target `3`, with starting weights `[1, 1]` and bias `0`. Now apply one plain SGD update with learning rate `0.05`, momentum `0`, and weight decay `0`.

The rule remains `new value = old value - learning rate × gradient`. The first weight has gradient zero and stays at `1`. The second weight has gradient `-4` and becomes `1.2`. Bias has gradient `-4` and becomes `0.2`.

Run this cell **once after Lesson 05**. The gradients are already calculated, so do not clear them before this step. To repeat this experiment, rerun Lesson 05 first to reset parameters and calculate fresh gradients. Repeating `step()` alone would reuse old gradients.

With plain SGD and these settings, zero gradient leaves a parameter unchanged. Momentum or weight decay can also contribute to updates; the conclusion here applies to this specific configuration.

**Before running:** predict the updated output for `[0, 1]` using weights `[1, 1.2]` and bias `0.2`. Is it closer to target `3` than the earlier prediction `1`?

**My prediction:** _write here_


In [ ]:
# Register all three parameter values owned by the two-input layer.
optimizer = torch.optim.SGD(training_model.parameters(), lr=0.05, momentum=0, weight_decay=0)
print("Weights before step:", training_model.weight.detach().flatten().tolist())
print("Bias before step:", training_model.bias.item())

optimizer.step()  # Use the gradients calculated in Lesson 05.

# Measure the updated prediction and loss with a fresh forward pass.
with torch.no_grad():
    updated_prediction = training_model(example_x)
    updated_loss = loss_fn(updated_prediction, example_y)

print("Weights after step:", training_model.weight.detach().flatten().tolist())
print("Bias after step:", training_model.bias.item())
print("Updated prediction:", updated_prediction.item())
print("Updated loss:", updated_loss.item())


**My observation and explanation:** _write here_

The updated prediction is closer to this example's supplied target, and its loss decreased. This is one update on one example; it does not measure performance on the full dataset or unseen examples.


<a id="batch-gradients"></a>
## Lesson 07: combine the two examples into one batch

Use the two examples together: `[1, 0]` with target `5`, and `[0, 1]` with target `3`. Reset weights to `[1, 1]` and bias to `0` so their starting parameters match our individual-example experiments.

`MSELoss()` averages the two squared errors. Its gradient for **each parameter** is the mean of that parameter's two example contributions. It does not average the gradients of different parameters together.

| Example | Weight 1 gradient | Weight 2 gradient | Bias gradient |
|---|---:|---:|---:|
| `[1, 0]` → target 5 | -8 | 0 | -8 |
| `[0, 1]` → target 3 | 0 | -4 | -4 |
| Mean across examples | -4 | -2 | -6 |

A zero contribution from one example does not erase a nonzero contribution from the other. Here both weight gradients are nonzero. In other data, opposing contributions could cancel, so a nonzero batch gradient is not guaranteed for every batch.

For plain SGD at the same positive learning rate, the gradient's sign determines the direction and its absolute size determines the update size. Here weight 1's gradient has size 4 and weight 2's has size 2, so weight 1 would receive the larger update.

**Before running:** predict the starting predictions, mean squared loss, and both weight gradients. Does this cell update the parameters?

**My prediction:** _write here_


In [ ]:
# Reset the starting parameters for comparison with the individual examples.
with torch.no_grad():
    training_model.weight.fill_(1.)
    training_model.bias.fill_(0.)

# Two examples, two input features each; one target for each example.
batch_x = torch.tensor([[1., 0.], [0., 1.]])
batch_y = torch.tensor([[5.], [3.]])

training_model.zero_grad(set_to_none=True)  # Clear the previous gradients.
batch_predictions = training_model(batch_x)
batch_loss = loss_fn(batch_predictions, batch_y)  # Mean of both squared errors.
batch_loss.backward()  # Calculate gradients from the combined loss; no update yet.

print("Input shape:", batch_x.shape)
print("Predictions:", batch_predictions.detach().flatten().tolist())
print("Mean batch loss:", batch_loss.item())
print("Weight gradients [weight1, weight2]:", training_model.weight.grad.flatten().tolist())
print("Bias gradient:", training_model.bias.grad.item())
print("Weights still:", training_model.weight.detach().flatten().tolist())
print("Bias still:", training_model.bias.item())


**My observation and explanation:** _write here_

Both weights now have a learning signal from this batch. Each gradient belongs to its own parameter; SGD will update all the registered parameters using their respective gradients.


<a id="batch-update"></a>
## Lesson 08: apply the batch update to all three parameters

Lesson 07 left weights `[1, 1]`, bias `0`, and gradients `[-4, -2]` for the weights and `-6` for bias. Plain SGD with learning rate `0.05` gives:

- Weight 1: `1 - 0.05 × (-4) = 1.2`.
- Weight 2: `1 - 0.05 × (-2) = 1.1`.
- Bias: `0 - 0.05 × (-6) = 0.3`.

Each parameter uses its own gradient. The larger absolute weight gradient produces the larger weight update at this learning rate.

Run this cell **once immediately after Lesson 07**. Do not clear the gradients between backward and the update. To repeat the experiment, rerun Lesson 07 first: it resets parameters and calculates fresh gradients. If you restarted the kernel, run the notebook's earlier cells in order to recreate its variables.

After the update, predict for the same batch again and measure a fresh mean loss. The previously calculated `batch_loss` still records the pre-update value; it does not change automatically when parameters change.

**Before running:** calculate the new prediction for `[1, 0]` and for `[0, 1]`. Compare them with their respective targets `5` and `3`.

**My predictions:** _write here_


In [ ]:
# Register the layer's weights and bias for one plain SGD update.
batch_optimizer = torch.optim.SGD(training_model.parameters(), lr=0.05, momentum=0, weight_decay=0)
print("Mean batch loss before update:", batch_loss.item())

batch_optimizer.step()  # Use the batch gradients from Lesson 07.

# A fresh forward pass measures the effect of the parameter update.
with torch.no_grad():
    updated_batch_predictions = training_model(batch_x)
    updated_batch_loss = loss_fn(updated_batch_predictions, batch_y)

print("Updated weights:", training_model.weight.detach().flatten().tolist())
print("Updated bias:", training_model.bias.item())
print("Updated predictions:", updated_batch_predictions.flatten().tolist())
print("Targets:", batch_y.flatten().tolist())
print("Updated mean batch loss:", updated_batch_loss.item())


**My observation and explanation:** _write here_

This update improved the mean loss on these two training examples. It is one training update, not proof that the model has learned the correct rule for unseen examples. We will bring the other examples back when continuing training.


<a id="full-training"></a>
## Lesson 09: train all three parameters on all four examples

Bring back all four examples from Lesson 03, with targets from our known rule `3 × input₁ + input₂ + 2`:

| Inputs | Target |
|---|---:|
| `[0, 0]` | 2 |
| `[1, 0]` | 5 |
| `[0, 1]` | 3 |
| `[2, 4]` | 12 |

For `[0, 0]`, prediction equals bias, and both weight gradients from that example are zero. The other examples also contribute to the mean loss and its parameter gradients.

Reset weights to `[1, 1]` and bias to `0` for this repeatable experiment. This deliberate reset happens once before the loop. During training, each epoch starts from the parameters produced by the previous update.

Use all four examples in each update, so batch size is four and each update is one epoch. Create plain SGD once before the loop. Each epoch clears gradients, predicts, measures mean squared loss, calculates fresh gradients, and updates all three parameters. Evaluate a fresh loss after updating.

We will run 100 epochs and print selected ones. The printed loss is the mean over **all four training examples after the update**. The known ideal parameters are weights `[3, 1]` and bias `2`; a finite run need not reach those values exactly. We are not evaluating validation or test data in this cell.

**Before running:** will weights be reset every epoch? Why must predictions and gradients be recalculated each epoch?

**My prediction and explanation:** _write here_


In [ ]:
# Reset once for this new experiment, rather than inside the training loop.
with torch.no_grad():
    training_model.weight.fill_(1.)
    training_model.bias.fill_(0.)

# All four examples have two features and one target each.
train_x = torch.tensor([[0., 0.], [1., 0.], [0., 1.], [2., 4.]])
train_y = torch.tensor([[2.], [5.], [3.], [12.]])
loss_fn = torch.nn.MSELoss()
training_optimizer = torch.optim.SGD(training_model.parameters(), lr=0.05, momentum=0, weight_decay=0)

# Record the loss before training for comparison.
with torch.no_grad():
    initial_training_loss = loss_fn(training_model(train_x), train_y).item()
print("Initial training loss:", initial_training_loss)

for epoch in range(1, 101):
    training_optimizer.zero_grad(set_to_none=True)  # Clear the previous gradients.
    train_predictions = training_model(train_x)    # Use the current parameters.
    training_loss = loss_fn(train_predictions, train_y)
    training_loss.backward()                       # Calculate fresh gradients.
    training_optimizer.step()                      # Update both weights and bias.

    # Measure a new loss after this update; do not reuse the pre-update loss.
    with torch.no_grad():
        loss_after = loss_fn(training_model(train_x), train_y).item()

    if epoch in (1, 2, 5, 10, 20, 50, 100):
        print(
            f"Epoch {epoch:3d} | weight1={training_model.weight[0, 0].item():.4f} | "
            f"weight2={training_model.weight[0, 1].item():.4f} | "
            f"bias={training_model.bias.item():.4f} | training loss={loss_after:.6f}"
        )

# Compare final training predictions with the supplied targets.
with torch.no_grad():
    final_training_predictions = training_model(train_x)
print("Final training predictions:", final_training_predictions.flatten().tolist())
print("Targets:", train_y.flatten().tolist())


**My observation and explanation:** _write here_

Look at the overall training loss and predictions, as well as the three parameter values. Parameters move together to reduce the mean loss across the dataset; a single example's zero gradient does not imply a zero gradient for the entire batch.


<a id="validation"></a>
## Lesson 10: evaluate unseen validation examples

The loss printed during Lesson 09 was training loss: it measured predictions on the four examples used for updates. Now evaluate two new examples without training on them:

| Validation inputs | Correct target from `3 × input₁ + input₂ + 2` |
|---|---:|
| `[1, 1]` | 6 |
| `[3, 2]` | 13 |

These are synthetic examples, so we know their correct answers from the rule used to construct the data. In a real supervised task, validation examples need their own correct labels or targets.

Run this cell after Lesson 09 to evaluate the model learned after 100 epochs. Recompute the current training loss and calculate validation loss with the same mean squared error function. Use `torch.no_grad()` for evaluation; do not call `backward()` or `optimizer.step()` on these validation examples. Parameters remain unchanged.

Training and validation losses are measured on different examples, so they need not match. These two validation examples give a small check; they do not establish performance on every unseen input. Validation can guide training choices. Keep test data separate for final evaluation, as we learned in Chapter 03.

**Before running:** which parameters should evaluation change? Should validation targets influence a training update in this cell?

**My prediction and explanation:** _write here_


In [ ]:
# New examples that were not included in the training data.
validation_x = torch.tensor([[1., 1.], [3., 2.]])
validation_y = torch.tensor([[6.], [13.]])

# Evaluate both datasets using the current learned parameters.
# There is no backward call or optimizer update here.
with torch.no_grad():
    current_training_loss = loss_fn(training_model(train_x), train_y)
    validation_predictions = training_model(validation_x)
    validation_loss = loss_fn(validation_predictions, validation_y)

print("Validation inputs:", validation_x.tolist())
print("Validation targets:", validation_y.flatten().tolist())
print("Validation predictions:", validation_predictions.flatten().tolist())
print("Training loss:", current_training_loss.item())
print("Validation loss:", validation_loss.item())
print("Weights still:", training_model.weight.detach().flatten().tolist())
print("Bias still:", training_model.bias.item())


**My observation and explanation:** _write here_

Compare predictions against the corresponding validation targets. A higher validation loss than training loss at one checkpoint is not enough by itself to establish overfitting. To study that pattern, compare training and validation performance throughout training.

## Documentation

- [PyTorch: Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)
- [PyTorch: Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)
- [PyTorch: MSELoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.MSELoss.html)

- [PyTorch: SGD](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html)
